In [24]:
# Data handling
import pandas as pd
import numpy as np

# Preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import PCA

# Model selection
from sklearn.model_selection import train_test_split, GridSearchCV,StratifiedKFold, cross_val_score

# Classifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
# Pipeline
from sklearn.pipeline import Pipeline

# Evaluation
from sklearn.metrics import classification_report,roc_auc_score

#saving
import joblib


In [4]:
#importing csv
df=pd.read_csv("bank-full.csv",sep=";")

In [5]:
#data summary
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


**DATA PREPROCESSING**

In [6]:
#target
df["y"]=df["y"].map({"yes":1,"no":0})
y=df["y"]

In [7]:
#FEATURES

X=df.drop(columns=["day","month","y"])

In [8]:
X.head()

,age,job,marital,education,default,balance,housing,loan,contact,duration,campaign,pdays,previous,poutcome
0,58,management,married,tertiary,no,2143,yes,no,unknown,261,1,-1,0,unknown
1,44,technician,single,secondary,no,29,yes,no,unknown,151,1,-1,0,unknown
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,76,1,-1,0,unknown
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,92,1,-1,0,unknown
4,33,unknown,single,unknown,no,1,no,no,unknown,198,1,-1,0,unknown


**Handling binary columns**

In [9]:
#replacing yes with 1 and no with 0

l=["default","housing","loan"]
for i in l:
    X[i]=X[i].map({"yes":1,"no":0})

**creating column transformer**

In [10]:
num_col=["balance","age","duration","campaign","pdays","previous"]
cut_col=["job","marital","education","contact","poutcome"]

In [11]:
#creating a column transformer

prepocess=ColumnTransformer(transformers=[
    ("num_col",StandardScaler(),num_col),
    ("cat_col",OneHotEncoder(drop='first', handle_unknown='ignore'),cut_col),
])

In [12]:
prepocess

ColumnTransformer(transformers=[('num_col', StandardScaler(),
                                 ['balance', 'age', 'duration', 'campaign',
                                  'pdays', 'previous']),
                                ('cat_col',
                                 OneHotEncoder(drop='first',
                                               handle_unknown='ignore'),
                                 ['job', 'marital', 'education', 'contact',
                                  'poutcome'])])

In [15]:
#creating a pipeline

pipeline = Pipeline(steps=[
    ('preprocessor', prepocess),
    ('classifier', LogisticRegression(
        max_iter=1000,random_state=42,n_jobs=-1,
        penalty= 'l1',class_weight="balanced",C= 0.1,solver='liblinear'))
])


In [16]:
pipeline

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num_col', StandardScaler(),
                                                  ['balance', 'age', 'duration',
                                                   'campaign', 'pdays',
                                                   'previous']),
                                                 ('cat_col',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['job', 'marital',
                                                   'education', 'contact',
                                                   'poutcome'])])),
                ('classifier',
                 LogisticRegression(C=0.1, class_weight='balanced',
                                    max_iter=1000, n_jobs=-1, penalty='l1',
                                    random_state=42, solver='liblinear'))])

In [17]:
#creating training and test sets

x_train,x_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

**TRAINING MY BASELINE MODEL WICH IS LOGISTIC REGRATION**

In [18]:
df["y"].value_counts(normalize=True)

y
0    0.883015
1    0.116985
Name: proportion, dtype: float64

In [25]:
# Fit baseline model
pipeline.fit(x_train, y_train)

# Evaluate
y_pred = pipeline.predict(x_test)
print("Classification Report:\n", classification_report(y_test, y_pred))
y_pred_proba = pipeline.predict_proba(x_test)[:,1] 
roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f" rou_auc is: {roc_auc}")


C:\Users\precious\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1271: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 2.
  warnings.warn(


Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.83      0.89      7952
           1       0.39      0.77      0.51      1091

    accuracy                           0.83      9043
   macro avg       0.68      0.80      0.70      9043
weighted avg       0.89      0.83      0.85      9043

 rou_auc is: 0.8819609914297887


In [93]:
#STRATIFIEDKFOLD

CV=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
score=cross_val_score(pipeline,x_train,y_train,cv=CV,scoring="roc_auc")
score.mean()

C:\Users\precious\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1271: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 2.
  warnings.warn(
C:\Users\precious\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1271: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 2.
  warnings.warn(
C:\Users\precious\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1271: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 2.
  warnings.warn(
C:\Users\precious\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1271: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 2.
  warnings.warn(
C:\Users\precious\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1271: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'li

np.float64(0.8425983317019051)

**HYPERTUNE MY BASELINE MODEL**

In [99]:
param_grid={
    "classifier__C":[0.01,0.1,1,10],
    "classifier__penalty":["l1","l2"],
    "classifier__solver":["liblinear"]

}
grid_search=GridSearchCV(pipeline,param_grid=param_grid,cv=CV,scoring='roc_auc')
grid_search.fit(x_train,y_train)

print(f"best params is : {grid_search.best_params_}")
print(f" best rou_auc is: {grid_search.best_score_}")

C:\Users\precious\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1271: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 2.
  warnings.warn(
C:\Users\precious\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1271: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 2.
  warnings.warn(
C:\Users\precious\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1271: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 2.
  warnings.warn(
C:\Users\precious\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1271: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 2.
  warnings.warn(
C:\Users\precious\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1271: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'li

best params is : {'classifier__C': 0.1, 'classifier__penalty': 'l1', 'classifier__solver': 'liblinear'}
 best rou_auc is: 0.8855352560983809


**COMPARING MY BASELINE MODEL WITH  OTHER TWO MODELS**

RANDOM FOREST 

In [26]:
#creating a pipeline

rm_pipeline = Pipeline(steps=[
    ('preprocessor', prepocess),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        min_samples_split=10,
        max_features="sqrt",
        max_depth=7,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1))
])
"""
{'classifier__class_weight': 'balanced', 'classifier__max_depth': 7, 'classifier__max_features':
        'sqrt', 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 10, 'classifier__n_estimators': 200}

""" 


"\n{'classifier__class_weight': 'balanced', 'classifier__max_depth': 7, 'classifier__max_features':\n        'sqrt', 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 10, 'classifier__n_estimators': 200}\n\n"

In [29]:
joblib.dump(rm_pipeline, "random_forest_model.pkl")


['random_forest_model.pkl']

In [27]:
rm_pipeline

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num_col', StandardScaler(),
                                                  ['balance', 'age', 'duration',
                                                   'campaign', 'pdays',
                                                   'previous']),
                                                 ('cat_col',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['job', 'marital',
                                                   'education', 'contact',
                                                   'poutcome'])])),
                ('classifier',
                 RandomForestClassifier(class_weight='balanced', max_depth=7,
                                        min_samples_leaf=2,
                                        min_samples_split=10, n_estimators=200,
                                        n_jobs=-1, random_state=42))])

In [142]:
#fitting the pipeline and getting rou
rm_pipeline.fit(x_train, y_train)

# Evaluate
y_pred = rm_pipeline.predict(x_test)
print("Classification Report:\n", classification_report(y_test, y_pred))
y_pred_proba = rm_pipeline.predict_proba(x_test)[:,1] 
roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f" rou_auc is: {roc_auc}")

Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.80      0.87      7952
           1       0.36      0.82      0.50      1091

    accuracy                           0.80      9043
   macro avg       0.66      0.81      0.69      9043
weighted avg       0.90      0.80      0.83      9043

 rou_auc is: 0.8891615619473024


In [161]:
#STRATIFIEDKFOLD

CV=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
score=cross_val_score(rm_pipeline,x_train,y_train,cv=CV,scoring="roc_auc")
score.mean()

np.float64(0.8948208631992827)

hypertunning random forest

In [136]:
rm_pipeline = Pipeline(steps=[
    ('preprocessor', prepocess),
    ('classifier', RandomForestClassifier(
        random_state=42,
        
        
    ))
])

#parameter grid
param_grid={
    "classifier__n_estimators":[100,200,300],
    "classifier__max_depth":[3,5,7],
    "classifier__min_samples_split":[2,5,10],
    "classifier__min_samples_leaf":[1,2,4] ,
    "classifier__max_features":["sqrt","log2"] ,
    "classifier__class_weight":[None,"balanced"] ,

}

In [137]:
grid_search=GridSearchCV(rm_pipeline,param_grid=param_grid,cv=CV,scoring='roc_auc',n_jobs=-1)
grid_search.fit(x_train,y_train)

print(f"best params is : {grid_search.best_params_}")
print(f" best rou_auc is: {grid_search.best_score_}")

best params is : {'classifier__class_weight': 'balanced', 'classifier__max_depth': 7, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 10, 'classifier__n_estimators': 200}
 best rou_auc is: 0.8948208631992827


In [31]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        45211 non-null  int64 
 1   job        45211 non-null  object
 2   marital    45211 non-null  object
 3   education  45211 non-null  object
 4   default    45211 non-null  int64 
 5   balance    45211 non-null  int64 
 6   housing    45211 non-null  int64 
 7   loan       45211 non-null  int64 
 8   contact    45211 non-null  object
 9   duration   45211 non-null  int64 
 10  campaign   45211 non-null  int64 
 11  pdays      45211 non-null  int64 
 12  previous   45211 non-null  int64 
 13  poutcome   45211 non-null  object
dtypes: int64(9), object(5)
memory usage: 4.8+ MB


XGBCLASSIFIER

In [158]:

xgb_pipeline = Pipeline(steps=[
    ('preprocessor', prepocess),
    ('classifier', XGBClassifier(
        n_estimators=200,
        learning_rate=0.1,
        #min_samples_split=2,
        max_depth=3,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        use_label_encoder=False,
        eval_metric="aucpr",
      
        
    ))
])


In [145]:
xgb_pipeline


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num_col', StandardScaler(),
                                                  ['balance', 'age', 'duration',
                                                   'campaign', 'pdays',
                                                   'previous']),
                                                 ('cat_col',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['job', 'marital',
                                                   'education', 'contact',
                                                   'poutcome'])])),
                ('classifier',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsampl...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.1,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=3, max_leaves=None,
                               min_child_weight=None, min_samples_leaf=1,
                               min_samples_split=2, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=100, ...))])

In [159]:
#fitting the pipeline and getting rou
xgb_pipeline.fit(x_train, y_train)

# Evaluate
y_pred = xgb_pipeline.predict(x_test)
print("Classification Report:\n", classification_report(y_test, y_pred))
y_pred_proba = xgb_pipeline.predict_proba(x_test)[:,1] 
roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f" rou_auc is: {roc_auc}")

C:\Users\precious\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:11:16] WARNING: C:\b\abs_d97hy_84m6\croot\xgboost-split_1749630932152\work\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Classification Report:
               precision    recall  f1-score   support

           0       0.92      0.97      0.95      7952
           1       0.65      0.39      0.49      1091

    accuracy                           0.90      9043
   macro avg       0.78      0.68      0.72      9043
weighted avg       0.89      0.90      0.89      9043

 rou_auc is: 0.8988402228218072


In [160]:
#STRATIFIEDKFOLD

CV=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
score=cross_val_score(xgb_pipeline,x_train,y_train,cv=CV,scoring="roc_auc")
score.mean()

C:\Users\precious\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:11:35] WARNING: C:\b\abs_d97hy_84m6\croot\xgboost-split_1749630932152\work\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\precious\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:12:11] WARNING: C:\b\abs_d97hy_84m6\croot\xgboost-split_1749630932152\work\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\precious\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:12:15] WARNING: C:\b\abs_d97hy_84m6\croot\xgboost-split_1749630932152\work\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\precious\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:12:18] WARNING: C:\b\abs_d97hy_84m6\croot\xgboost-split_1749630932152\work\

np.float64(0.9021629011244029)

 hypertunning xgb

In [150]:
#pipeline
xgb_pipeline = Pipeline(steps=[
    ('preprocessor', prepocess),
    ('classifier', XGBClassifier( 
        random_state=42,
        use_label_encoder=False,
        eval_metric="logloss"
        
    ))
])

#parameter grid
xgb_param_grid={
    "classifier__n_estimators":[100,200,300],
    "classifier__max_depth":[3,5,7],
    "classifier__learning_rate":[0.01,0.1,0.2],
    "classifier__subsample":[0.8,1.0] ,
    "classifier__colsample_bytree":[0.8,1.0] ,
    "classifier__scale_pos_weight":[1,5,10] ,

}

In [151]:
#grid_search
grid_search=GridSearchCV(xgb_pipeline,param_grid=xgb_param_grid,cv=CV,scoring='roc_auc',n_jobs=-1)
grid_search.fit(x_train,y_train)

print(f"best params is : {grid_search.best_params_}")
print(f" best rou_auc is: {grid_search.best_score_}")

C:\Users\precious\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:04:38] WARNING: C:\b\abs_d97hy_84m6\croot\xgboost-split_1749630932152\work\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


best params is : {'classifier__colsample_bytree': 0.8, 'classifier__learning_rate': 0.1, 'classifier__max_depth': 3, 'classifier__n_estimators': 200, 'classifier__scale_pos_weight': 1, 'classifier__subsample': 0.8}
 best rou_auc is: 0.9020483820969639


#### **STACKING CLASSIFIER** 

In [165]:
from sklearn.ensemble import StackingClassifier

#base models
estimators=[
    ("rf", RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        min_samples_split=10,
        max_features="sqrt",
        max_depth=7,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1)),
    ("xgb", XGBClassifier(
        n_estimators=200,
        learning_rate=0.1,
        #min_samples_split=2,
        max_depth=3,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        use_label_encoder=False,
        eval_metric="logloss"))
]

# satking layer

stacking_clf=StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(
        max_iter=1000,random_state=42,n_jobs=-1,
        penalty= 'l1',class_weight="balanced",C= 0.1,solver='liblinear'),
    cv=CV
)

#PIPE LINE

#pipeline
stack_pipeline = Pipeline(steps=[
    ('preprocessor', prepocess),
    ('classifier',stacking_clf )
])

#EVALUATION

CV=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
score=cross_val_score(stack_pipeline,x_train,y_train,cv=CV,scoring="roc_auc")
score.mean()

C:\Users\precious\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:26:41] WARNING: C:\b\abs_d97hy_84m6\croot\xgboost-split_1749630932152\work\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\precious\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:27:29] WARNING: C:\b\abs_d97hy_84m6\croot\xgboost-split_1749630932152\work\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\precious\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:27:32] WARNING: C:\b\abs_d97hy_84m6\croot\xgboost-split_1749630932152\work\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\precious\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:27:34] WARNING: C:\b\abs_d97hy_84m6\croot\xgboost-split_1749630932152\work\

np.float64(0.8979918266543125)

In [166]:
stack_pipeline.fit(x_train,y_train)

C:\Users\precious\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:33:39] WARNING: C:\b\abs_d97hy_84m6\croot\xgboost-split_1749630932152\work\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\precious\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:34:57] WARNING: C:\b\abs_d97hy_84m6\croot\xgboost-split_1749630932152\work\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\precious\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:34:59] WARNING: C:\b\abs_d97hy_84m6\croot\xgboost-split_1749630932152\work\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\precious\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:35:02] WARNING: C:\b\abs_d97hy_84m6\croot\xgboost-split_1749630932152\work\

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num_col', StandardScaler(),
                                                  ['balance', 'age', 'duration',
                                                   'campaign', 'pdays',
                                                   'previous']),
                                                 ('cat_col',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['job', 'marital',
                                                   'education', 'contact',
                                                   'poutcome'])])),
                ('classifier',
                 StackingClassifier(cv=StratifiedKFold(n_splits=5, random_state=42, s...
                                                               max_cat_to_onehot=None,
                                                               max_delta_step=None,
                                                               max_depth=3,
                                                               max_leaves=None,
                                                               min_child_weight=None,
                                                               missing=nan,
                                                               monotone_constraints=None,
                                                               multi_strategy=None,
                                                               n_estimators=200,
                                                               n_jobs=None,
                                                               num_parallel_tree=None, ...))],
                                    final_estimator=LogisticRegression(C=0.1,
                                                                       class_weight='balanced',
                                                                       max_iter=1000,
                                                                       n_jobs=-1,
                                                                       penalty='l1',
                                                                       random_state=42,
                                                                       solver='liblinear')))])

In [167]:
pred=stack_pipeline.predict(x_test)

y_pred = stack_pipeline.predict(x_test)
print("Classification Report:\n", classification_report(y_test, y_pred))
y_pred_proba = stack_pipeline.predict_proba(x_test)[:,1] 
roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f" rou_auc is: {roc_auc}")

Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.81      0.88      7952
           1       0.37      0.81      0.50      1091

    accuracy                           0.81      9043
   macro avg       0.67      0.81      0.69      9043
weighted avg       0.90      0.81      0.83      9043

 rou_auc is: 0.89341295250882


In [1]:
X.info()

NameError: name 'X' is not defined